In [ ]:
# Preprocess FaceForensics++ Dataset
#Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# UPLOAD "kaggle.json" to use kaggle API

from google.colab import files
files.upload()  # upload kaggle.json


In [ ]:
# Set Kaggle crendentials

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download the Public Dataset

DATASET = "xdxd003/ff-c23"

!kaggle datasets download -d $DATASET -p ./dataset

In [ ]:
# To User the preprocessing scripts

!git clone https://github.com/akshaymina/SFANet.git

In [ ]:
# Setup Variables and Paths for prep.

DRIVE = "/content/drive/MyDrive"
ZIP_FFPP   = "./dataset/ff-c23.zip"      # your FF++ zip path
#ZIP_CELEB  = f"{DRIVE}/celeb-df-v2.zip"           # your Celeb-DF zip path
#ZIP_WILD   = f"{DRIVE}/wild-deepfake.zip"      # your WildDeepfake zip path

WORK = "/content/sfanet_data"   # working dir (fast local disk)
# OUT  = f"{DRIVE}/sfanet_preprocessed"  # save final output to Drive
OUT  = "/content/sfanet_preprocessed"

# Demo mode: set to 5 for quick test, None for full dataset
MAX_VIDEOS = None    # videos for each category
MAX_FRAMES = 25      # faces extracted per video
FRAME_STRIDE = 20    # process every Nth frame
COMPRESSION = "c23"  # FF++ quality: raw | c23 | c40

In [ ]:
# Path creation and resolution

import os, sys, shutil, zipfile, subprocess
from pathlib import Path

sys.path.insert(0, "/content/SFANet/preprocessing")
Path(WORK).mkdir(parents=True, exist_ok=True)
Path(OUT).mkdir(parents=True, exist_ok=True)

print("Ready.")

In [ ]:
# Extract the Zip file

from pathlib import Path
import subprocess

RAW_FFPP = Path(f"{WORK}/raw/ffpp")
RAW_FFPP.mkdir(parents=True, exist_ok=True)


print(f"Extracting {ZIP_FFPP} -> {RAW_FFPP}")

subprocess.run(
    [
        "unzip",
        "-q",
        ZIP_FFPP,
        "-d",
        str(RAW_FFPP)
    ],
    check=True
)

print("Extraction completed.")

print("Dataset location:", RAW_FFPP)

In [ ]:
# EXTRACT FACES FROM VIDEOS

!pip install --no-deps facenet-pytorch
!pip install retina-face

import sys

print("--- Checking System & Core Libraries ---")
print(f"Python version: {sys.version}")

# 1. Verify facenet-pytorch
try:
    from facenet_pytorch import MTCNN, InceptionResnetV1
    import torch

    # Initialize the model structure to verify weight configurations
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

    print("\n✅ facenet-pytorch: SUCCESS")
    print(f"   - Running on device: {device}")
except Exception as e:
    print(f"\n❌ facenet-pytorch: FAILED\n   Error: {e}")

# 2. Verify retina-face
try:
    from retinaface import RetinaFace

    # Initialize the model blueprint to ensure imports work seamlessly
    model = RetinaFace.build_model()

    print("\n✅ retina-face: SUCCESS")
    print("   - Core model structure built successfully.")
except Exception as e:
    print(f"\n❌ retina-face: FAILED\n   Error: {e}")


In [ ]:
# Include Partially Processed Dataset from your Google Drive

import os
import shutil

# Define paths
DRIVE_ZIP_PATH = "/content/drive/MyDrive/sfanet-processed-01.zip"
COLAB_ZIP_PATH = "/content/sfanet-processed-01.zip"
UNZIP_DIR = "/content/temp_unzip"
FINAL_DEST_DIR = "/content/sfanet_data/processed/"

os.makedirs(FINAL_DEST_DIR, exist_ok=True)

print(f"Copying {DRIVE_ZIP_PATH} to {COLAB_ZIP_PATH}...")
shutil.copy(DRIVE_ZIP_PATH, COLAB_ZIP_PATH)
print("Copy complete.")

# Ensure the unzip directory exists and clear it if it contains previous extractions
if os.path.exists(UNZIP_DIR):
    shutil.rmtree(UNZIP_DIR)
os.makedirs(UNZIP_DIR)

print(f"Unzipping {COLAB_ZIP_PATH} to {UNZIP_DIR}...")
shutil.unpack_archive(COLAB_ZIP_PATH, UNZIP_DIR)
print("Unzip complete.")

# Construct the path to the 'ffpp' folder based on user's description
# The zip contains 'sfanet-processed-01/ffpp'
UNZIPPED_FFPP_PATH = os.path.join(UNZIP_DIR, "sfanet-processed-01", "ffpp")

if os.path.exists(UNZIPPED_FFPP_PATH):
    print(f"Moving {UNZIPPED_FFPP_PATH} to {FINAL_DEST_DIR}...")
    # Use distutils.dir_util.copy_tree to merge contents if ffpp already exists
    # or shutil.move to move the directory directly if it doesn't.
    # For simplicity, if ffpp exists in destination, we will overwrite/merge.
    # More robust would be to copy content or check for existing files.
    # Let's move the 'ffpp' folder into 'FINAL_DEST_DIR'

    # If 'ffpp' already exists in FINAL_DEST_DIR, we need to remove it first or merge.
    # Given the prompt, the user wants 'ffpp' inside FINAL_DEST_DIR, so let's move the folder.
    target_ffpp_path = os.path.join(FINAL_DEST_DIR, "ffpp")
    if os.path.exists(target_ffpp_path):
        print(f"Target folder {target_ffpp_path} already exists. Removing before move...")
        shutil.rmtree(target_ffpp_path)
    shutil.move(UNZIPPED_FFPP_PATH, FINAL_DEST_DIR)
    print(f"'ffpp' folder moved to {FINAL_DEST_DIR} successfully.")
else:
    print(f"Error: Expected folder '{UNZIPPED_FFPP_PATH}' not found after unzipping.")

# Clean up temporary files and directories
print("Cleaning up temporary files...")
os.remove(COLAB_ZIP_PATH)
shutil.rmtree(UNZIP_DIR)
print("Cleanup complete.")
print("Operation finished.")

In [ ]:
# Face Extraction

from pathlib import Path
import subprocess
import sys

sys.path.insert(0, "./preprocessing") # Updated path
Path(WORK).mkdir(parents=True, exist_ok=True)
Path(OUT).mkdir(parents=True, exist_ok=True)

FFPP_RAW = "/content/sfanet_data/raw/ffpp/FaceForensics++_C23"

print("FF++ raw root:", FFPP_RAW)

if not Path(FFPP_RAW).exists():
    raise FileNotFoundError(
        f"Dataset root not found: {FFPP_RAW}"
    )

print("\nRaw dataset folders:")
for p in sorted(Path(FFPP_RAW).iterdir()):
    if p.is_dir():
        print(" -", p.name)

FFPP_OUT = f"{WORK}/processed/ffpp"

cmd = [
    sys.executable,
    "-u",  # unbuffered Python output

    "./preprocessing/preprocess_ffpp.py", # Updated script path

    "--raw_root",
    FFPP_RAW,

    "--out_root",
    FFPP_OUT,

    "--compression",
    COMPRESSION,

    "--frame_stride",
    str(FRAME_STRIDE),

    "--max_frames",
    str(MAX_FRAMES),
]

if MAX_VIDEOS:
    cmd += [
        "--max_videos",
        str(MAX_VIDEOS)
    ]

print("\nRunning:")
print(" ".join(cmd))
print("-" * 80)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Stream output live
for line in process.stdout:
    print(line, end="", flush=True)

process.wait()

print("\n" + "-" * 80)
print("Exit code:", process.returncode)

if process.returncode != 0:
    raise RuntimeError(
        f"Preprocessing failed with code {process.returncode}"
    )

print("Preprocessing completed successfully.")

In [ ]:
# Archieve and Download the prep Dataset

import shutil
from google.colab import files
import os

output_filename = 'sfanet_processed_data'
archive_name = shutil.make_archive(output_filename, 'zip', '/content/sfanet_data/processed')

files.download(archive_name)

In [ ]:
# Move processed data (Zip) to drive

!rm -rf /content/sfanet_data/raw/*

!cp /content/sfanet_processed_data.zip /content/drive/MyDrive/

In [ ]:
# VERIFICATION

In [ ]:
from pathlib import Path
from collections import defaultdict

OUT = Path("/content/sfanet_data/processed/ffpp")

print("Checking processed output...\n")

for folder in [
    "original_sequences",
    "manipulated_sequences/Deepfakes",
    "manipulated_sequences/Face2Face",
    "manipulated_sequences/FaceSwap",
    "manipulated_sequences/NeuralTextures",
]:
    root = OUT / folder

    if not root.exists():
        print(f"{folder:45} MISSING")
        continue

    video_dirs = [
        p for p in root.rglob("*")
        if p.is_dir() and any(p.glob("*.png"))
    ]

    png_count = sum(1 for _ in root.rglob("*.png"))

    print(
        f"{folder:45} "
        f"video_dirs={len(video_dirs):5d} | "
        f"images={png_count:7d}"
    )